In [12]:
import math
import os
import re
import sys
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import torchaudio
from torchvision import transforms
from torchvision.io import read_video
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


In [ ]:
# Paths and defaults

SCRIPT_DIR = Path.cwd()
REPO_ROOT= Path("/home/amin/Research/Representation/Movie/cav-mae-sync")

SRC_PATH = REPO_ROOT / "src"



print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH, "exists=", SRC_PATH.exists())
print("models dir exists=", (SRC_PATH / "models").exists())
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
from models import CAVMAESync  # type: ignore
DEFAULT_TOTAL_FRAMES = 16
DEFAULT_AUDIO_MEAN = -5.081
DEFAULT_AUDIO_STD = 4.4849
DEFAULT_SEG_SEC = 4.0
DEFAULT_MODEL_PATH = REPO_ROOT / "pretrained_models" / "cav_mae_sync.pth"
DEFAULT_OUTPUTS_ROOT = Path ("/home/amin/Research/Representation/Movie/outputs/cav-mae-sync")


REPO_ROOT: /home/amin/Research/Representation/Movie/movie_watching/cav-mae-sync
SRC_PATH: /home/amin/Research/Representation/Movie/movie_watching/cav-mae-sync/src exists= True
models dir exists= True


In [14]:
def run_ffmpeg(video_path: Path, ffmpeg_path: str = "ffmpeg") -> Path:
    """Extract mono 16 kHz audio; returns path to wav."""
    wav_path = video_path.with_suffix(".wav")
    cmd = [
        ffmpeg_path,
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-y",
        str(wav_path),
    ]
    print(f"[ffmpeg] {video_path.name} -> {wav_path.name}")
    import subprocess

    subprocess.run(cmd, check=True)
    return wav_path


def map_frame_to_spectrogram(frame_index: int, num_frames: int, spectrogram_length: int, target_length: int):
    """Mirror dataloader_sync mapping from frame idx to spectrogram window."""
    frame_position = int(round(frame_index * spectrogram_length / num_frames))
    start = max(0, frame_position - target_length // 2)
    end = start + target_length
    if end > spectrogram_length:
        end = spectrogram_length
        start = max(0, end - target_length)
    return start, end


def guess_target_length(seg_sec: float) -> int:
    """Heuristic to match repo defaults (multiples of 16 used in retrieval script)."""
    lookup = {
        2: 192,
        3: 304,
        4: 416,
        5: 512,
        6: 624,
        7: 720,
        10: 1024,
    }
    rounded = int(round(seg_sec))
    if rounded in lookup:
        return lookup[rounded]
    approx = seg_sec / 10.0 * 1024.0
    return int(math.ceil(max(approx, 1) / 16) * 16)


def load_waveform(audio_path: Path) -> Tuple[torch.Tensor, int, float]:
    """Load full audio once; return mono waveform, sample_rate, duration."""
    try:
        waveform, sample_rate = torchaudio.load(audio_path)
    except Exception as e:
        raise RuntimeError(
            f"Failed to load audio from {audio_path} with torchaudio ({e}). "
            "Extract the audio to a WAV and rerun."
        )

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform - waveform.mean()
    num_samples = waveform.shape[1]
    duration = num_samples / sample_rate
    return waveform, sample_rate, duration


def slice_waveform(waveform: torch.Tensor, sample_rate: int, start_s: float, end_s: float) -> torch.Tensor:
    """Slice preloaded waveform to [start, end) seconds."""
    start = int(start_s * sample_rate)
    end = int(end_s * sample_rate)
    start = max(start, 0)
    end = min(end, waveform.shape[1])
    if end <= start:
        return waveform[:, :1].clone()
    return waveform[:, start:end]


def waveform_to_logmel(
    waveform: torch.Tensor,
    sample_rate: int,
    mel_bins: int = 128,
) -> torch.Tensor:
    """Compute log-mel filterbank and pad/truncate to 1024 like dataloader_sync."""
    fbank = torchaudio.compliance.kaldi.fbank(
        waveform,
        htk_compat=True,
        sample_frequency=sample_rate,
        use_energy=False,
        window_type="hanning",
        num_mel_bins=mel_bins,
        dither=0.0,
        frame_shift=10,
    )
    base_length = 1024
    if fbank.shape[0] < base_length:
        pad = base_length - fbank.shape[0]
        fbank = F.pad(fbank, (0, 0, 0, pad))
    elif fbank.shape[0] > base_length:
        fbank = fbank[:base_length]
    return fbank


def build_audio_windows(fbank: torch.Tensor, num_frames: int, target_length: int, audio_mean: float, audio_std: float):
    """Slice spectrogram into per-frame windows and normalize."""
    specs = []
    for frame_idx in range(num_frames):
        start, end = map_frame_to_spectrogram(frame_idx, num_frames, fbank.shape[0], target_length)
        window = fbank[start:end, :]
        if window.shape[0] < target_length:
            window = F.pad(window, (0, 0, 0, target_length - window.shape[0]))
        window = (window - audio_mean) / audio_std
        specs.append(window)
    return torch.stack(specs)


def load_video_frames(
    *,
    video_path: Path,
    start_s: float,
    end_s: float,
    num_frames: int,
    transform,
) -> torch.Tensor:
    """Decode frames from the mp4 and sample uniformly within [start_s, end_s)."""
    frames, _, _ = read_video(str(video_path), start_pts=start_s, end_pts=end_s, pts_unit="sec")
    if frames.numel() == 0:
        raise RuntimeError(f"No frames decoded between {start_s} and {end_s} seconds")
    indices = torch.linspace(0, frames.shape[0] - 1, steps=num_frames).round().clamp(0, frames.shape[0] - 1).long()
    imgs = []
    for idx in indices:
        frame = frames[idx].numpy()
        img = Image.fromarray(frame)
        imgs.append(transform(img))
    return torch.stack(imgs)


def load_model(
    model_path: Path,
    device: torch.device,
    target_length: int,
    total_frames: int,
    num_register_tokens: Optional[int] = None,
):
    state = torch.load(model_path, map_location=device)
    cleaned = {k.replace("module.", "", 1): v for k, v in state.items()}

    if num_register_tokens is None and "register_tokens" in cleaned:
        num_register_tokens = cleaned["register_tokens"].shape[0] // 2
        print(f"Auto-detected num_register_tokens={num_register_tokens} from checkpoint")

    model = CAVMAESync(
        audio_length=target_length,
        modality_specific_depth=11,
        num_register_tokens=num_register_tokens if num_register_tokens is not None else 4,
        cls_token=True,
        total_frame=total_frames,
    )
    msg = model.load_state_dict(cleaned, strict=False)
    print("Loaded model:", msg)
    print(f"Checkpoint keys: {len(cleaned)}; model keys: {len(model.state_dict())}; "
          f"missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")
    for name in [
        "patch_embed_a.proj.weight",
        "patch_embed_v.proj.weight",
        "blocks_a.0.attn.qkv.weight",
    ]:
        if name in model.state_dict():
            t = model.state_dict()[name]
            t = t.float()
            print(f"param {name} norm={t.norm().item():.4g} min={t.min().item():.4g} max={t.max().item():.4g}")
    model.to(device)
    model.eval()
    return model


def get_transform():
    return transforms.Compose(
        [
            transforms.Resize(224, interpolation=Image.BICUBIC),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )


def run_segment(
    model,
    waveform: torch.Tensor,
    sample_rate: int,
    start_s: float,
    end_s: float,
    target_length: int,
    total_frames: int,
    video_path: Path,
    device: torch.device,
    transform,
):
    """Compute per-modality embeddings for a [start,end) segment."""
    segment_waveform = slice_waveform(waveform, sample_rate, start_s, end_s)
    fbank = waveform_to_logmel(segment_waveform, sample_rate)
    audio_windows = build_audio_windows(fbank, total_frames, target_length, DEFAULT_AUDIO_MEAN, DEFAULT_AUDIO_STD)
    video_frames = load_video_frames(
        start_s=start_s,
        end_s=end_s,
        num_frames=total_frames,
        transform=transform,
        video_path=video_path,
    )

    if start_s == 0.0:
        print("debug audio_windows min/max", float(audio_windows.min()), float(audio_windows.max()))
        print("debug audio_windows mean/std", float(audio_windows.mean()), float(audio_windows.std()))
        print("debug video_frames min/max", float(video_frames.min()), float(video_frames.max()))
        print("debug video_frames mean/std", float(video_frames.mean()), float(video_frames.std()))
    audio_in = audio_windows.to(device)
    video_in = video_frames.to(device)
    with torch.no_grad():
        _, _, cls_a, cls_v = model.forward_feat(audio_in, video_in)
        if start_s == 0.0:
            print("debug cls_a min/max", float(cls_a.min()), float(cls_a.max()), "norm", float(cls_a.norm()))
            print("debug cls_v min/max", float(cls_v.min()), float(cls_v.max()), "norm", float(cls_v.norm()))

    if cls_a.ndim == 1:
        cls_a = cls_a.unsqueeze(0)
    if cls_v.ndim == 1:
        cls_v = cls_v.unsqueeze(0)

    audio_embed = cls_a.view(total_frames, -1).mean(dim=0)
    video_embed = cls_v.view(total_frames, -1).mean(dim=0)
    return (
        audio_embed.cpu().numpy().astype(np.float32),
        video_embed.cpu().numpy().astype(np.float32),
    )


In [15]:
def prepare_model_bundle(
    *,
    seg_sec: float,
    target_length: Optional[int],
    total_frames: int,
    num_register_tokens: Optional[int],
    model_path: Path,
    device: str,
):
    if target_length is None or target_length == 0:
        target_length = guess_target_length(seg_sec)

    torch_device = torch.device("cuda" if device == "cuda" and torch.cuda.is_available() else "cpu")
    print("Using device:", torch_device)
    print(f"Target log-mel length: {target_length}")

    model = load_model(Path(model_path), torch_device, target_length, total_frames, num_register_tokens)
    transform = get_transform()

    return {
        "model": model,
        "device": torch_device,
        "target_length": target_length,
        "transform": transform,
        "total_frames": total_frames,
    }


def extract_embeddings_for_video(
    *,
    video_path: Path,
    audio_path: Optional[Path],
    outdir: Path,
    seg_sec: float,
    bundle: dict,
):
    audio_path = audio_path or video_path.with_suffix(".wav")
    outdir.mkdir(parents=True, exist_ok=True)

    waveform, sample_rate, duration = load_waveform(audio_path)
    print(f"Loaded audio (sr={sample_rate}) duration={duration:.2f}s")

    audio_embeddings = []
    video_embeddings = []

    num_segments = int(math.ceil(duration / seg_sec))
    for idx in range(num_segments):
        start_s = idx * seg_sec
        end_s = min(duration, (idx + 1) * seg_sec)
        if end_s <= start_s:
            continue
        print(f"Processing segment {idx + 1}/{num_segments}: {start_s:.2f}s-{end_s:.2f}s")
        audio_embed, video_embed = run_segment(
            model=bundle["model"],
            waveform=waveform,
            sample_rate=sample_rate,
            start_s=start_s,
            end_s=end_s,
            target_length=bundle["target_length"],
            total_frames=bundle["total_frames"],
            video_path=video_path,
            device=bundle["device"],
            transform=bundle["transform"],
        )
        audio_embeddings.append(audio_embed)
        video_embeddings.append(video_embed)

    if not audio_embeddings:
        raise RuntimeError("No embeddings were produced; check video path and decoding support.")

    audio_arr = np.stack(audio_embeddings, axis=0)
    video_arr = np.stack(video_embeddings, axis=0)

    np.save(outdir / "embeddings_audio.npy", audio_arr)
    np.save(outdir / "embeddings_video.npy", video_arr)
    print("Saved embeddings_audio.npy with shape", audio_arr.shape)
    print("Saved embeddings_video.npy with shape", video_arr.shape)

    return audio_arr, video_arr


def extract_embeddings_for_pairs(
    *,
    pairs: List[Tuple[Path, Path]],
    outdir: Path,
    seg_sec: float,
    bundle: dict,
):
    outdir.mkdir(parents=True, exist_ok=True)

    all_audio = []
    all_video = []

    for idx, (video_path, audio_path) in enumerate(pairs, start=1):
        print(f"[{idx}/{len(pairs)}] {video_path.name}")
        waveform, sample_rate, duration = load_waveform(audio_path)
        end_s = min(duration, seg_sec) if seg_sec else duration
        audio_embed, video_embed = run_segment(
            model=bundle["model"],
            waveform=waveform,
            sample_rate=sample_rate,
            start_s=0.0,
            end_s=end_s,
            target_length=bundle["target_length"],
            total_frames=bundle["total_frames"],
            video_path=video_path,
            device=bundle["device"],
            transform=bundle["transform"],
        )
        all_audio.append(audio_embed)
        all_video.append(video_embed)

    if not all_audio:
        raise RuntimeError("No embeddings were produced; check chunk paths and decoding support.")

    audio_arr = np.stack(all_audio, axis=0)
    video_arr = np.stack(all_video, axis=0)

    np.save(outdir / "embeddings_audio.npy", audio_arr)
    np.save(outdir / "embeddings_video.npy", video_arr)
    print("Saved embeddings_audio.npy with shape", audio_arr.shape)
    print("Saved embeddings_video.npy with shape", video_arr.shape)

    return audio_arr, video_arr


In [16]:
def extract_part_number(filename: Path):
    """Extract part number from filename like 'video_part_001.mp4'."""
    match = re.search(r"part[_-]?(\d+)", filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None


def collect_chunk_pairs(base_dir: Path, run_range=range(1, 19)):
    """Collect paired 2s video/audio chunks matching the peav_small naming scheme."""
    all_pairs = []
    for run_idx in run_range:
        vid_chunk_dir = base_dir / f"Video{run_idx}" / f"Video{run_idx}_chunks"
        aud_chunk_dir = base_dir / f"Audio{run_idx}" / f"Audio{run_idx}_chunks"

        if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
            print(f"Warning: Missing dir for run {run_idx}")
            continue

        vids = sorted(vid_chunk_dir.glob("*.mp4"), key=lambda p: extract_part_number(p) or -1)
        auds = sorted(aud_chunk_dir.glob("*.wav"), key=lambda p: extract_part_number(p) or -1)
        if len(vids) == 0 or len(auds) == 0:
            print(f"Warning: No chunks found for run {run_idx}")
            continue

        vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}

        common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))
        for part_num in common_parts:
            all_pairs.append((vid_parts[part_num], aud_parts[part_num]))

        print(f"Run {run_idx}: {len(common_parts)} paired chunks")

    print(f"Total paired 2s segments: {len(all_pairs)}")
    return all_pairs


In [17]:
def run_cavmae_wrapper(
    *,
    segmented: bool,
    videos: Optional[List[Path]] = None,
    outputs_root: Path = DEFAULT_OUTPUTS_ROOT,
    seg_sec: float = DEFAULT_SEG_SEC,
    target_length: Optional[int] = None,
    total_frames: int = DEFAULT_TOTAL_FRAMES,
    num_register_tokens: Optional[int] = None,
    model_path: Path = DEFAULT_MODEL_PATH,
    device: str = "cpu",
    ffmpeg_path: str = "ffmpeg",
    chunks_base_dir: Optional[Path] = None,
):
    """
    Wrapper to run CAV-MAE Sync.

    If segmented=False: process full videos into 4s segments (like the scripts).
    If segmented=True: use 2s pre-chunked audio/video pairs like peav_small.ipynb.
    """
    outputs_root = Path(outputs_root).expanduser().resolve()
    outputs_root.mkdir(parents=True, exist_ok=True)

    if segmented:
        if chunks_base_dir is None:
            raise ValueError("chunks_base_dir is required when segmented=True")
        seg_sec = 2.0
        pairs = collect_chunk_pairs(Path(chunks_base_dir).expanduser().resolve())
        if not pairs:
            raise RuntimeError("No paired chunks found; check chunks_base_dir")

        outdir = outputs_root / "segmented_2s"
        bundle = prepare_model_bundle(
            seg_sec=seg_sec,
            target_length=target_length,
            total_frames=total_frames,
            num_register_tokens=num_register_tokens,
            model_path=model_path,
            device=device,
        )
        extract_embeddings_for_pairs(
            pairs=pairs,
            outdir=outdir,
            seg_sec=seg_sec,
            bundle=bundle,
        )
        return outdir

    if not videos:
        raise ValueError("videos is required when segmented=False")

    bundle = prepare_model_bundle(
        seg_sec=seg_sec,
        target_length=target_length,
        total_frames=total_frames,
        num_register_tokens=num_register_tokens,
        model_path=model_path,
        device=device,
    )

    for vid in videos:
        video_path = Path(vid).expanduser().resolve()
        if not video_path.exists():
            raise FileNotFoundError(f"Video not found: {video_path}")

        run_ffmpeg(video_path, ffmpeg_path)
        outdir = outputs_root / video_path.stem
        extract_embeddings_for_video(
            video_path=video_path,
            audio_path=None,
            outdir=outdir,
            seg_sec=seg_sec,
            bundle=bundle,
        )

    return outputs_root


In [18]:
# --- Official dataloader + retrieval-style extraction ---
import json
from torch.utils.data import DataLoader
import dataloader_sync
from dataloader_sync import train_collate_fn


def write_minimal_label_csv(label_csv_path: Path, labels):
    """Create a minimal label CSV for the sync dataloader."""
    label_csv_path.parent.mkdir(parents=True, exist_ok=True)
    with open(label_csv_path, "w") as f:
        f.write("mid,index,display_name")
        for idx, mid in enumerate(labels):
            f.write(f"{mid},{idx},{mid}")


def write_sync_dataset_json(json_path: Path, wav_path: Path, video_id: str, frames_dir: Path, labels):
    """Write a minimal dataset JSON expected by dataloader_sync."""
    json_path.parent.mkdir(parents=True, exist_ok=True)
    data = {
        "data": [
            {
                "wav": str(wav_path),
                "labels": ",".join(labels),
                "video_id": video_id,
                "video_path": str(frames_dir),
            }
        ]
    }
    json_path.write_text(json.dumps(data, indent=2))


def get_wav_duration_s(wav_path: Path) -> float:
    waveform, sr = torchaudio.load(wav_path)
    return float(waveform.shape[1]) / float(sr)


def extract_frames_for_sync(
    *,
    video_path: Path,
    frames_root: Path,
    video_id: str,
    total_frames: int,
    duration_s: float,
    ffmpeg_path: str = "ffmpeg",
):
    """Extract frames into frame_*/VIDEO_ID.jpg for dataloader_sync."""
    import subprocess
    frames_dir = frames_root / video_id
    temp_dir = frames_dir / "_tmp_frames"
    temp_dir.mkdir(parents=True, exist_ok=True)

    fps = total_frames / max(duration_s, 1e-6)
    cmd = [
        ffmpeg_path,
        "-i",
        str(video_path),
        "-vf",
        f"fps={fps}",
        "-vframes",
        str(total_frames),
        "-q:v",
        "2",
        str(temp_dir / "frame_%03d.jpg"),
    ]
    print("[ffmpeg] extracting frames:", " ".join(cmd))
    subprocess.run(cmd, check=True)

    frame_files = sorted(temp_dir.glob("frame_*.jpg"))
    if len(frame_files) < total_frames:
        raise RuntimeError(f"Expected {total_frames} frames, got {len(frame_files)}")

    for idx, src in enumerate(frame_files[:total_frames]):
        frame_dir = frames_dir / f"frame_{idx}"
        frame_dir.mkdir(parents=True, exist_ok=True)
        dst = frame_dir / f"{video_id}.jpg"
        src.replace(dst)

    for extra in temp_dir.glob("*"):
        extra.unlink()
    temp_dir.rmdir()
    return frames_dir


def extract_embeddings_official_sync(
    *,
    video_path: Path,
    outputs_root: Path,
    seg_sec: float,
    model_path: Path,
    device: str = "cpu",
    total_frames: int = DEFAULT_TOTAL_FRAMES,
    num_register_tokens: Optional[int] = None,
    cls_token: bool = True,
    aggregation: str = "mean",
    ffmpeg_path: str = "ffmpeg",
):
    """Use dataloader_sync + retrieval-style forward to extract embeddings."""
    video_path = Path(video_path).expanduser().resolve()
    outputs_root = Path(outputs_root).expanduser().resolve()
    outputs_root.mkdir(parents=True, exist_ok=True)

    video_id = video_path.stem
    wav_path = video_path.with_suffix(".wav")
    if not wav_path.exists():
        run_ffmpeg(video_path, ffmpeg_path)

    duration_s = get_wav_duration_s(wav_path)
    target_length = guess_target_length(seg_sec)
    print(f"Target log-mel length: {target_length}")

    # Prepare frames for the sync dataloader
    frames_root = outputs_root / "frames_sync"
    frames_dir = extract_frames_for_sync(
        video_path=video_path,
        frames_root=frames_root,
        video_id=video_id,
        total_frames=total_frames,
        duration_s=duration_s,
        ffmpeg_path=ffmpeg_path,
    )

    # Build minimal dataset files
    labels = ["m0"]
    label_csv = outputs_root / "labels_dummy.csv"
    data_json = outputs_root / f"{video_id}_sync.json"
    write_minimal_label_csv(label_csv, labels)
    write_sync_dataset_json(data_json, wav_path, video_id, frames_dir, labels)

    audio_conf = {
        "num_mel_bins": 128,
        "target_length": target_length,
        "freqm": 0,
        "timem": 0,
        "mixup": 0,
        "dataset": "custom",
        "mode": "retrieval",
        "mean": DEFAULT_AUDIO_MEAN,
        "std": DEFAULT_AUDIO_STD,
        "noise": False,
        "im_res": 224,
        "frame_use": 5,
        "num_samples": 1,
        "total_frame": total_frames,
    }

    dataset = dataloader_sync.AudiosetDataset(
        str(data_json),
        audio_conf=audio_conf,
        label_csv=str(label_csv),
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=train_collate_fn)

    torch_device = torch.device("cuda" if device == "cuda" and torch.cuda.is_available() else "cpu")
    model = load_model(model_path, torch_device, target_length, total_frames, num_register_tokens)
    model.eval()

    all_audio = []
    all_video = []
    with torch.no_grad():
        for batch in loader:
            a_input, v_input, labels, video_ids, frame_indices = batch
            # flatten frames like retrieval.py
            a_input = a_input.reshape(a_input.shape[0] * a_input.shape[1], a_input.shape[2], a_input.shape[3])
            v_input = v_input.reshape(a_input.shape[0], v_input.shape[2], v_input.shape[3], v_input.shape[4])

            audio_input, video_input = a_input.to(torch_device), v_input.to(torch_device)
            if cls_token:
                tokens_a, tokens_v, cls_a, cls_v = model.forward_feat(audio_input, video_input)
                audio_output = cls_a
                video_output = cls_v
            else:
                audio_output, video_output = model.forward_feat(audio_input, video_input)
                audio_output = torch.mean(audio_output, dim=1)
                video_output = torch.mean(video_output, dim=1)

            audio_output = F.normalize(audio_output, dim=-1)
            video_output = F.normalize(video_output, dim=-1)

            # group back to (batch, frames, dim)
            num_frames = total_frames
            audio_output = audio_output.view(len(video_ids), num_frames, -1).cpu().numpy().astype(np.float32)
            video_output = video_output.view(len(video_ids), num_frames, -1).cpu().numpy().astype(np.float32)

            all_audio.append(audio_output)
            all_video.append(video_output)

    audio_arr = np.concatenate(all_audio, axis=0)
    video_arr = np.concatenate(all_video, axis=0)

    if aggregation == "mean":
        audio_agg = audio_arr.mean(axis=1)
        video_agg = video_arr.mean(axis=1)
    elif aggregation == "diagonal":
        # use per-frame embeddings as-is for sync; no extra aggregation
        audio_agg = audio_arr
        video_agg = video_arr
    else:
        raise ValueError(f"Unknown aggregation: {aggregation}")

    outdir = outputs_root / video_id / "official_sync"
    outdir.mkdir(parents=True, exist_ok=True)
    np.save(outdir / "embeddings_audio_frames.npy", audio_arr)
    np.save(outdir / "embeddings_video_frames.npy", video_arr)
    np.save(outdir / "embeddings_audio.npy", audio_agg)
    np.save(outdir / "embeddings_video.npy", video_agg)

    print("Saved embeddings_audio_frames.npy", audio_arr.shape)
    print("Saved embeddings_video_frames.npy", video_arr.shape)
    print("Saved embeddings_audio.npy", audio_agg.shape)
    print("Saved embeddings_video.npy", video_agg.shape)

    return outdir


In [19]:
# 4s segments
outputs = run_cavmae_wrapper(
    segmented=False,
    videos=[Path("/home/amin/Research/Representation/Movie/data/7T_MOVIE1_CC1_v2.mp4")],
    outputs_root=DEFAULT_OUTPUTS_ROOT,
    seg_sec=4.0,
    model_path=DEFAULT_MODEL_PATH,
    device="cpu",
)  


Using device: cpu
Target log-mel length: 416
Auto-detected num_register_tokens=8 from checkpoint
A CAV-MAE Model
Use norm_pix_loss:  False
Learnable Positional Embedding:  False
Keep Register Tokens:  False
Number of Audio Patches: 208, Visual Patches: 196
Number of Registers: 8
Using CLS Token
Using 16 frames
Audio Positional Embedding Shape: torch.Size([1, 208, 768])
Visual Positional Embedding Shape: torch.Size([1, 196, 768])
Loaded model: _IncompatibleKeys(missing_keys=['constrative_head_audio.0.norm1.weight', 'constrative_head_audio.0.norm1.bias', 'constrative_head_audio.0.norm1_a.weight', 'constrative_head_audio.0.norm1_a.bias', 'constrative_head_audio.0.norm1_v.weight', 'constrative_head_audio.0.norm1_v.bias', 'constrative_head_audio.0.attn.qkv.weight', 'constrative_head_audio.0.attn.qkv.bias', 'constrative_head_audio.0.attn.proj.weight', 'constrative_head_audio.0.attn.proj.bias', 'constrative_head_audio.0.norm2.weight', 'constrative_head_audio.0.norm2.bias', 'constrative_head_a

/home/amin/miniconda3/envs/cav-mae-sync/lib/python3.7/site-packages/torchvision/transforms/transforms.py:330: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  "Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. "
ffmpeg version 5.1.2 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 12.3.0 (conda-forge gcc 12.3.0-2)
  configuration: --prefix=/home/amin/miniconda3/envs/cav-mae-sync --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1696213708285/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1696213708285/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1696213708285/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1696213708285/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --disable-opens

Loaded audio (sr=16000) duration=921.02s
Processing segment 1/231: 0.00s-4.00s
debug audio_windows min/max -2.421767473220825 1.1329126358032227
debug audio_windows mean/std -0.3261299431324005 1.7486073970794678
debug video_frames min/max -2.1179039478302 2.640000104904175
debug video_frames mean/std -1.9282585382461548 0.48518386483192444
debug cls_a min/max -3.5694453716278076 3.2887160778045654 norm 73.05027770996094
debug cls_v min/max -2.6320927143096924 5.643736839294434 norm 74.5176010131836
Processing segment 2/231: 4.00s-8.00s
Processing segment 3/231: 8.00s-12.00s
Processing segment 4/231: 12.00s-16.00s
Processing segment 5/231: 16.00s-20.00s
Processing segment 6/231: 20.00s-24.00s
Processing segment 7/231: 24.00s-28.00s
Processing segment 8/231: 28.00s-32.00s
Processing segment 9/231: 32.00s-36.00s
Processing segment 10/231: 36.00s-40.00s
Processing segment 11/231: 40.00s-44.00s
Processing segment 12/231: 44.00s-48.00s
Processing segment 13/231: 48.00s-52.00s
Processing seg

In [21]:
import numpy as np

# Load the embeddings you just saved
audio = np.load("/home/amin/Research/Representation/Movie/outputs/7T_MOVIE1_CC1_v2/cav-mae-sync/embeddings_audio.npy")
video = np.load("/home/amin/Research/Representation/Movie/outputs/7T_MOVIE1_CC1_v2/cav-mae-sync/embeddings_video.npy")

print("Audio shape:", audio.shape)
print("Audio min/max:", audio.min(), audio.max())
print("Audio mean/std:", audio.mean(), audio.std())
print("Audio all zeros?", np.allclose(audio, 0))

print("\nVideo shape:", video.shape)
print("Video min/max:", video.min(), video.max())
print("Video mean/std:", video.mean(), video.std())
print("Video all zeros?", np.allclose(video, 0))


Audio shape: (231, 768)
Audio min/max: -3.169529 2.7559154
Audio mean/std: -0.00033807717 0.49373078
Audio all zeros? False

Video shape: (231, 768)
Video min/max: -3.9272513 5.6437373
Video mean/std: 0.0077848625 0.56331766
Video all zeros? False


In [ ]:
# # 2s pre-chunked segments
# BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Data/Separated")
# outputs = run_cavmae_wrapper(
#     segmented=True,
#     chunks_base_dir=BASE_DIR,
#     outputs_root=DEFAULT_OUTPUTS_ROOT,
#     model_path=DEFAULT_MODEL_PATH,
#     device="cuda",
# )
